# PharmaRL — GRPO training on SARS-CoV-2 Mpro env

**What this does**: trains Qwen3-1.5B to design Mpro inhibitor candidates by editing SELFIES strings. Talks to a deployed PharmaRL OpenEnv (HF Space) over HTTP. Logs reward curves to W&B.

**Runtime**: Colab T4 (free) is enough thanks to Unsloth 4-bit + LoRA.

**Env URL**: edit `ENV_URL` below to point at your HF Space.

## 1. Install

In [ ]:
!pip install -q unsloth
!pip install -q --upgrade --no-cache-dir 'trl>=0.11.0' 'transformers>=4.40.0' 'peft' 'accelerate'
!pip install -q openenv-core[core] selfies rdkit-pypi PyTDC wandb

## 2. Config

In [ ]:
import os

ENV_URL = os.environ.get('PHARMARL_ENV_URL', 'http://localhost:8000')  # set to HF Space URL once deployed
MODEL_NAME = 'unsloth/Qwen2.5-1.5B-Instruct'  # swap to Qwen3 once Unsloth supports it
MAX_SEQ_LEN = 1024
LORA_RANK = 16
NUM_GENERATIONS = 8           # G in GRPO — must be ≥2
MAX_TRAINING_STEPS = 500
WANDB_PROJECT = 'pharmarl'
WANDB_RUN_NAME = 'mpro-trivial-curriculum-v1'

## 3. Load model with Unsloth (4-bit + LoRA)

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
    fast_inference=True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=LORA_RANK,
    use_gradient_checkpointing='unsloth',
    random_state=42,
)

## 4. Connect to env

In [ ]:
import requests
import uuid

# Smoke test: hit the env once. NOTE: episode_id required — the env keeps state across calls per id.
_eid = str(uuid.uuid4())
r = requests.post(f'{ENV_URL}/reset', json={'difficulty': 'trivial', 'episode_id': _eid}).json()
print('reset OK:', r['observation']['smiles'], '   episode_id:', r['observation']['episode_id'])
print('vocab:', r['observation']['available_fragments'])


## 5. Reward functions (GRPO-style — multi-component)

GRPO accepts a list of reward functions; each maps `(completion, env_response)` to a float. We separate them so W&B graphs each component independently.

In [ ]:
import re
import json

_JSON_RE = re.compile(r'\{[^{}]*\}')

def parse_action(text: str):
    for m in _JSON_RE.findall(text):
        try:
            return json.loads(m)
        except Exception:
            continue
    return None

def reward_format(prompts, completions, **kwargs):
    """Was the model output a parseable JSON action?"""
    return [0.5 if parse_action(c) is not None else -0.5 for c in completions]

def reward_action_valid(prompts, completions, env_responses=None, **kwargs):
    """Did the env accept the action?"""
    out = []
    for env_r in (env_responses or [None] * len(completions)):
        if env_r is None:
            out.append(0.0)
            continue
        out.append(0.1 if env_r['observation'].get('last_action_valid') else -0.1)
    return out

def reward_env(prompts, completions, env_responses=None, **kwargs):
    """Pass through the env's reward (the main signal)."""
    out = []
    for env_r in (env_responses or [None] * len(completions)):
        out.append(float(env_r['reward']) if env_r else 0.0)
    return out

## 6. Rollout helper — interact with env to produce trajectories

TRL's GRPOTrainer expects single-turn (prompt → completion → reward). For multi-step env interaction we wrap it: each rollout = full episode, the "completion" is the concatenated action history, the reward is cumulative.

In [ ]:
import uuid

SYSTEM = '''You design SARS-CoV-2 Mpro inhibitors by editing SELFIES molecules.
Respond with ONE JSON action per turn. Allowed: ADD_FRAGMENT, REMOVE_FRAGMENT, SUBSTITUTE_ATOM, TERMINATE.'''

def rollout_episode(model, tokenizer, env_url, difficulty='trivial', max_new_tokens=80, max_steps=30):
    """One full episode against the env.

    The env is multi-step and keeps state per episode_id — we generate a fresh id for
    each rollout so concurrent GRPO group members do not collide.
    """
    episode_id = str(uuid.uuid4())
    obs = requests.post(
        f'{env_url}/reset',
        json={'difficulty': difficulty, 'episode_id': episode_id},
    ).json()['observation']
    actions, rewards, env_responses = [], [], []
    cumulative = 0.0
    for _ in range(max_steps):
        prompt = (
            f'{SYSTEM}\n\nSMILES: {obs["smiles"]}\n'
            f'Fragments: {obs["available_fragments"][:8]}\n'
            f'Valid actions: {obs["valid_actions"]}\n'
            'Respond with JSON action:'
        )
        inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, temperature=0.7, do_sample=True)
        txt = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
        action = parse_action(txt) or {'action_type': 'ADD_FRAGMENT', 'fragment': 'C', 'position': 0}
        # IMPORTANT: /step takes {'action': <action>, 'episode_id': <eid>}, NOT the raw action.
        step = requests.post(
            f'{env_url}/step',
            json={'action': action, 'episode_id': episode_id},
        ).json()
        actions.append(action)
        rewards.append(step['reward'])
        env_responses.append(step)
        cumulative += step['reward']
        obs = step['observation']
        if step['done']:
            break
    return {
        'actions': actions,
        'rewards': rewards,
        'env_responses': env_responses,
        'cumulative': cumulative,
        'final_smiles': obs['smiles'],
        'episode_id': episode_id,
    }


## 7. Smoke run (5 episodes BEFORE training)

In [ ]:
FastLanguageModel.for_inference(model)
baseline = [rollout_episode(model, tokenizer, ENV_URL, 'trivial') for _ in range(5)]
import statistics
print(f'Baseline avg cumulative reward: {statistics.mean(r["cumulative"] for r in baseline):.3f}')
for i, r in enumerate(baseline):
    print(f'  ep{i+1}: cum={r["cumulative"]:.3f}  final={r["final_smiles"]}')

## 8. GRPO training loop (custom — TRL's GRPOTrainer is single-turn; we drive episodes manually)

In [ ]:
import torch
import wandb
wandb.init(project=WANDB_PROJECT, name=WANDB_RUN_NAME)

FastLanguageModel.for_training(model)
optim = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=5e-6)

from server.curriculum import pick_difficulty  # if running with PharmaRL repo on path

for step in range(MAX_TRAINING_STEPS):
    difficulty = pick_difficulty(step) if False else (
        'trivial' if step < 100 else ('easy' if step < 300 else 'hard')
    )
    # Sample G=NUM_GENERATIONS rollouts in parallel (the 'group' in GRPO)
    FastLanguageModel.for_inference(model)
    rollouts = [rollout_episode(model, tokenizer, ENV_URL, difficulty) for _ in range(NUM_GENERATIONS)]
    rewards = [r['cumulative'] for r in rollouts]
    mean_r = sum(rewards) / len(rewards)
    advantages = [(r - mean_r) for r in rewards]

    # NOTE: full GRPO update requires re-running the model in train mode on the action tokens,
    # weighted by advantages, with a KL penalty against the reference. This stub logs the
    # group statistics and writes them to W&B; replace with TRL GRPOTrainer.train_step
    # once you've adapted the multi-turn rollout into a flat (prompt, completion) batch.

    wandb.log({
        'step': step,
        'difficulty': difficulty,
        'mean_reward': mean_r,
        'max_reward': max(rewards),
        'min_reward': min(rewards),
        'reward_std': torch.tensor(rewards).std().item(),
    })
    print(f'step={step} {difficulty:8s} mean={mean_r:.3f} max={max(rewards):.3f}')

## 9. Save trained model to HF Hub (asks for HF token; do NOT auto-push without confirming)

In [ ]:
# UNCOMMENT after training, and only after you confirm the run:
# from huggingface_hub import login
# login()  # paste token interactively
# model.push_to_hub('anshumanatrey/pharmarl-qwen-trained')
# tokenizer.push_to_hub('anshumanatrey/pharmarl-qwen-trained')

## 10. After-training smoke run (compare to baseline)

Run the same 5 episodes again, compare cumulative reward. This is the demo number for the video.

In [ ]:
FastLanguageModel.for_inference(model)
trained = [rollout_episode(model, tokenizer, ENV_URL, 'trivial') for _ in range(5)]
print(f'Trained avg cumulative reward: {statistics.mean(r["cumulative"] for r in trained):.3f}')
for i, r in enumerate(trained):
    print(f'  ep{i+1}: cum={r["cumulative"]:.3f}  final={r["final_smiles"]}')